<a href="https://colab.research.google.com/github/zerogradlabs/zerograd-python/blob/main/examples/evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Evaluate and decide: honest splits, the right threshold, the columns that matter

A championship tells you which model wins on one holdout. Before a model goes to production you want three more answers:

1. **Does it hold on other splits?** Above all when rows repeat per customer or the model will predict the future.
2. **Where should the score cut?** 0.5 is rarely what the business wants.
3. **Which columns carry it?**

This notebook answers all three on a loan book with the open source SDK, locally: no account, no API key. At the end, the same models inside scikit-learn.

In [ ]:
!pip install -q zerograd

## The data

A synthetic loan book: 600 customers with several loans each over two years. Default depends on the debt ratio, income and the
customer's own history (some customers are simply riskier), and the risk drifts up over time. Those two traits are what a random split hides.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(0)
n_customers, n = 600, 4000
customer = rng.integers(0, n_customers, n)
risk = rng.normal(0, 1, n_customers)[customer]                     # each customer's own risk
date = pd.Timestamp("2024-01-01") + pd.to_timedelta(rng.integers(0, 730, n), unit="D")
months = (date - date.min()).days / 30
ratio = rng.beta(2, 5, n)
income = rng.lognormal(10.5, 0.5, n).round(-2)
purpose = rng.choice(["car", "home", "studies", "other"], n, p=[0.3, 0.3, 0.1, 0.3])
logit = -3 + 4 * ratio - 0.6 * (np.log(income) - 10.5) + 0.9 * risk + 0.03 * months + 0.5 * (purpose == "other")
loans = pd.DataFrame({
    "customer_id": customer, "date": date, "debt_ratio": ratio.round(3), "income": income, "purpose": purpose,
    "default": (rng.random(n) < 1 / (1 + np.exp(-logit))).astype(int),
})
loans.head()

In [ ]:
loans["default"].mean().round(3), loans["customer_id"].value_counts().mean().round(1)

## 1. Cross-validation, three ways

`zg.cross_validate` runs every installed model on each fold and scores a baseline next to them. Binary tasks also get KS and lift in the top 10%.

In [ ]:
from zerograd import ZeroGrad

zg = ZeroGrad(models=["tabpfn-v2", "tabicl-v2"])   # leave models out to run every installed model
data = loans.drop(columns=["customer_id", "date"])
random_cv = zg.cross_validate(data, "default", folds=5)
random_cv.summary

Now split **by customer**: every loan of a customer lands on the same side, so a model is never scored on someone it already read in its
context. This is the number to trust when the model will score new customers.

In [ ]:
by_customer = zg.cross_validate(loans.drop(columns=["date"]), "default", folds=5, groups="customer_id")
by_customer.summary

And **by time**: train on the past, score the next block, walk forward. This is the number to trust when the model will score next
month's applications. `gap=` leaves rows out between the two sides, for targets that take a while to be known.

In [ ]:
by_time = zg.cross_validate(loans.drop(columns=["customer_id"]), "default", folds=4, time="date")
by_time.summary

In [ ]:
pd.DataFrame({
    "random": random_cv.summary["roc_auc"],
    "by customer": by_customer.summary["roc_auc"],
    "by time": by_time.summary["roc_auc"],
}).round(3)

`.folds` has one row per model and fold, if you want the spread behind the mean.

In [ ]:
by_time.folds[["model", "fold", "train_rows", "test_rows", "roc_auc", "ks", "lift_at_10"]].round(3)

## 2. Pick the threshold

Train a capability on everything but the last quarter, then pick the cut. Say a missed default costs 20 times what a false alarm costs (a
manual review). `optimize_threshold` reads the confusion counts the championship saved for every cut, so it needs no new predictions.

In [ ]:
cut = loans["date"] < "2025-10-01"
train, recent = loans[cut].drop(columns=["customer_id", "date"]), loans[~cut].drop(columns=["customer_id", "date"])
cap = zg.create("loan-default", examples=train, target="default")

In [ ]:
by_cost = cap.optimize_threshold("cost", cost_fp=1, cost_fn=20)
by_cost

In [ ]:
by_cost.table.loc[[0.1, 0.2, 0.3, 0.5]].round(3)

Other objectives: the most recall while 8 in 10 flags are right, the best F1, or Youden's J. `apply=False` only looks.

In [ ]:
for objective, kw in [("recall", {"min_precision": 0.5}), ("f1", {}), ("youden", {})]:
    try:
        print(cap.optimize_threshold(objective, apply=False, **kw))
    except Exception as e:  # a constraint no cut meets says so
        print(objective, "->", e)

The capability now predicts with the cost-optimal cut. Check it on the quarter it never saw:

In [ ]:
from sklearn.metrics import confusion_matrix

preds = cap.predict(recent.drop(columns="default"))
print("threshold", cap.threshold)
confusion_matrix(recent["default"], preds["prediction"])

## 3. Which columns matter

Shuffle one column at a time in rows the model did not read, and see how much ROC AUC drops.

In [ ]:
cap.importance(recent, repeats=3)

## 4. Inside scikit-learn

`ZeroGradClassifier` and `ZeroGradRegressor` are scikit-learn estimators: `fit` keeps a context, `predict_proba` runs the model with it. They
work in pipelines, grid searches and every scikit-learn tool that takes an estimator. You pick the model; there is no championship inside.

In [ ]:
from sklearn.model_selection import GroupKFold, cross_val_score

from zerograd import ZeroGradClassifier

X, y = loans[["debt_ratio", "income", "purpose"]], loans["default"]
clf = ZeroGradClassifier(model="tabicl-v2")
cross_val_score(clf, X, y, groups=loans["customer_id"], cv=GroupKFold(5), scoring="roc_auc").round(3)

In [ ]:
from sklearn.inspection import permutation_importance

clf.fit(X[cut], y[cut])
r = permutation_importance(clf, X[~cut], y[~cut], scoring="roc_auc", n_repeats=3, random_state=0)
pd.Series(r.importances_mean, index=X.columns).sort_values(ascending=False).round(4)

## Next

* `zg.context_curve(...)` finds the smallest context that scores as well, for faster predictions.
* The [no target notebook](https://github.com/zerogradlabs/zerograd-python/blob/main/examples/no-target.ipynb) covers anomalies, imputation, synthetic rows and clustering.
* Docs: [zerograd.dev/docs#evaluation](https://zerograd.dev/docs#evaluation).